In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('/content/drive/MyDrive/car-trader/car-trader-dataset/car-data.csv')

#
columns_to_keep = ['brand', 'year', 'mileage_km', 'fuel_type', 'transmission', 'price_usd']

df = df[columns_to_keep]

# print
df.head()

,brand,year,mileage_km,fuel_type,transmission,price_usd
0,Chevrolet,2011,185945,Hybrid,Manual,5903
1,Toyota,2017,141520,Plug-in Hybrid,Automatic,9277
2,BMW,2016,139091,Gasoline,Automatic,18918
3,Honda,2007,258093,Gasoline,Automatic,5058
4,Hyundai,2017,147560,Hybrid,Automatic,16954


In [5]:
#convert from usd to rwf
conversion_rate = 1458.29
df['price_usd'] = df['price_usd'] * conversion_rate
df['price_usd'] = df['price_usd'].astype('int64')

# renaming columns to proper names
df.rename(columns={
    'price_usd': 'price-rwf',
    'brand': 'car-maker',
    'mileage_km': 'mileage-km',
    'fuel_type': 'fuel-type',
    'transmission': 'transmission-type',

}, inplace=True)

# lowecase all string columns
df = df.apply(lambda x: x.str.lower() if x.dtype == 'object' else x)

df.head()

,car-maker,year,mileage-km,fuel-type,transmission-type,price-rwf
0,chevrolet,2011,185945,hybrid,manual,8608285
1,toyota,2017,141520,plug-in hybrid,automatic,13528556
2,bmw,2016,139091,gasoline,automatic,27587930
3,honda,2007,258093,gasoline,automatic,7376030
4,hyundai,2017,147560,hybrid,automatic,24723848


In [6]:
remove_brands = ['chevrolet', 'subaru', 'mazda']
df = df[~df['car-maker'].isin(remove_brands)]

#
df['car-maker'].unique()

array(['toyota', 'bmw', 'honda', 'hyundai', 'ford', 'volkswagen', 'audi',
       'tesla', 'nissan', 'mercedes-benz', 'jeep', 'kia'], dtype=object)

In [7]:
df['fuel-type'] = df['fuel-type'].replace('plug-in hybrid', 'hybrid')

df['fuel-type'].unique()

array(['hybrid', 'gasoline', 'electric', 'diesel'], dtype=object)

In [8]:
df['transmission-type'].unique()

array(['automatic', 'manual'], dtype=object)

In [9]:


# 1. BASIC OVERVIEW
print(f"\n Dataset Shape: {df.shape}")
print(f"\n Missing Values:\n{df.isnull().sum()}")
print(f"\n Duplicates: {df.duplicated().sum()}")


 Dataset Shape: (8448686, 6)

 Missing Values:
car-maker            0
year                 0
mileage-km           0
fuel-type            0
transmission-type    0
price-rwf            0
dtype: int64

 Duplicates: 16428


In [10]:
# remove duplicates
df = df.drop_duplicates()

In [11]:
# 3. OUTLIER DETECTION (IQR Method)

numerical_cols = ['price-rwf', 'mileage-km']

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"\n{col}:")
    print(f"  Outliers: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")
    print(f"  Min: {df[col].min():,.0f} | Max: {df[col].max():,.0f}")
    print(f"  99th percentile: {df[col].quantile(0.99):,.0f}")


price-rwf:
  Outliers: 473687 (5.62%)
  Min: 1,166,632 | Max: 364,572,500
  99th percentile: 87,659,270

mileage-km:
  Outliers: 0 (0.00%)
  Min: 0 | Max: 500,000
  99th percentile: 438,983


In [12]:
# 4. SANITY CHECKS
print(f"Prices = 0: {(df['price-rwf'] == 0).sum()}")
print(f"Negative prices: {(df['price-rwf'] < 0).sum()}")
print(f"Mileage > 500k km: {(df['mileage-km'] > 500000).sum()}")

Prices = 0: 0
Negative prices: 0
Mileage > 500k km: 0


In [13]:
df.to_csv('/content/drive/MyDrive/car-trader/car-trader-dataset/cars-data-cleaned.csv', index=False)